# Phi-3.5 Agent - Inference & Testing

This notebook provides a complete environment to test your fine-tuned Phi-3.5 agent. It handles:
1. **Special Tokens**: Verifying `<tool_use>`, `<tool_name>`, and `<parameters>` are correctly loaded.
2. **Chat Templates**: Using the Phi-3.5 instruct template to match training.
3. **Inference**: Generating tool calls or text responses based on user queries.

In [ ]:
import torch
import logging
from pathlib import Path
from peft import PeftModel
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    BitsAndBytesConfig,
    pipeline
)
from transformers.utils import logging as hf_logging

# Setup logging
logging.basicConfig(level=logging.INFO)
log = logging.getLogger("AgentTest")
hf_logging.set_verbosity_error()

def patch_phi3_dynamic_cache():
    """Fixes compatibility between Phi-3 and newer Transformers versions."""
    try:
        from transformers.cache_utils import DynamicCache
        if not hasattr(DynamicCache, "seen_tokens"):
            DynamicCache.seen_tokens = property(lambda self: self.get_seq_length(0))
        if not hasattr(DynamicCache, "get_max_length"):
            def get_max_length(self, layer_idx: int = 0) -> int: return self.get_max_cache_shape(layer_idx)
            DynamicCache.get_max_length = get_max_length
        if not hasattr(DynamicCache, "get_usable_length"):
            def get_usable_length(self, new_seq_length: int, layer_idx: int = 0) -> int:
                max_length = self.get_max_cache_shape(layer_idx)
                prev = self.get_seq_length(layer_idx)
                if max_length is not None and max_length > 0 and prev + new_seq_length > max_length:
                    return max_length - new_seq_length
                return prev
            DynamicCache.get_usable_length = get_usable_length
    except Exception as e:
        log.warning(f"DynamicCache patch failed: {e}")

patch_phi3_dynamic_cache()

In [ ]:
# --- CONFIGURATION ---
BASE_MODEL_ID = "microsoft/Phi-3.5-mini-instruct"
ADAPTER_PATH = r"D:\self projects\FileWise-Agent\MCP\SLM_Agent\SLM_Agent_Github\slm-agent\models\phi3-agent-final"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if (DEVICE == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16

TOOL_SPECIAL_TOKENS = [
    "<tool_use>",
    "</tool_use>",
    "<tool_name>",
    "</tool_name>",
    "<parameters>",
    "</parameters>",
]

In [ ]:
print(f"Loading tokenizer from {ADAPTER_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)

# Verify special tokens
for token in TOOL_SPECIAL_TOKENS:
    tid = tokenizer.convert_tokens_to_ids(token)
    print(f"Token {token:15} -> ID {tid}")
    if tid == tokenizer.unk_token_id:
        print(f"WARNING: Token {token} is UNK!")

print(f"Loading base model {BASE_MODEL_ID} in 4-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=DTYPE,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Resize embeddings to match tokenizer (crucial for added tokens)
base_model.resize_token_embeddings(len(tokenizer))

print(f"Loading LoRA adapter from {ADAPTER_PATH}...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print("Model loaded and ready for inference.")

In [ ]:
def get_system_prompt():
    return """You are an AI assistant that can use tools to help solve problems. 
You have access to the following tools:

- web_search(query: str, max_results: int = 5): Search the internet for real-time information.
- file_reader(file_path: str, operation: str = 'read'): Read content from local files.

To use a tool, wrap the call in XML-like tags:
<tool_use>
<tool_name>name_of_tool</tool_name>
<parameters>
{"param_name": "value"}
</parameters>
</tool_use>"""

def generate_response(user_input, max_new_tokens=512, temperature=0.1):
    messages = [
        {"role": "system", "content": get_system_prompt()},
        {"role": "user", "content": user_input}
    ]
    
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else 1.0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    # Decode only the generated part
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=False)
    new_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    
    # We keep special tokens for tool calls visualization if they exist in the raw output
    raw_new_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    
    return raw_new_text.strip()

### Run Automated Tests
See how the model handles standard queries.

In [ ]:
test_queries = [
    "What's the current weather in New York?",
    "Read the file 'config.json' and tell me the version number.",
    "Who won the Oscar for Best Picture in 2025?",
    "Hello! How are you today?"
]

for query in test_queries:
    print(f"\nUSER: {query}")
    response = generate_response(query)
    print(f"AGENT:\n{response}")
    print("-" * 50)

### Interactive Test
Type your own queries below.

In [ ]:
from IPython.display import display, Markdown
import ipywidgets as widgets

def on_submit(b):
    user_input = text_input.value
    if not user_input.strip(): return
    
    with output_area:
        print(f"User: {user_input}")
        response = generate_response(user_input)
        print(f"Agent: {response}\n")
    text_input.value = ""

text_input = widgets.Text(placeholder='Type your query here...', layout={'width': '80%'})
send_button = widgets.Button(description='Send')
output_area = widgets.Output()

send_button.on_click(on_submit)
display(widgets.HBox([text_input, send_button]), output_area)